In [ ]:
#Muhammad Ibtihaj
#SP23-BAI-037

import pandas as pd
import numpy as np
import re
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("/content/mbti_1.csv")
print("Dataset loaded:", df.shape)
print(df.head())

-
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\|{2,}', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_posts'] = df['posts'].apply(clean_text)


df['IE'] = df['type'].apply(lambda x: 1 if x[0]=='I' else 0)
df['NS'] = df['type'].apply(lambda x: 1 if x[1]=='N' else 0)
df['TF'] = df['type'].apply(lambda x: 1 if x[2]=='T' else 0)
df['JP'] = df['type'].apply(lambda x: 1 if x[3]=='J' else 0)


vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
X = vectorizer.fit_transform(df['clean_posts'])
print("TF-IDF shape:", X.shape)

joblib.dump(vectorizer, "vectorizer.pkl")

traits = ['IE', 'NS', 'TF', 'JP']
models = {}

for trait in traits:
    print(f"\nTraining for {trait}...")
    y = df[trait]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=300, class_weight='balanced', solver='lbfgs')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{trait} Accuracy: {acc:.3f}")
    print(classification_report(y_test, y_pred))
    joblib.dump(model, f"{trait}_model.pkl")
    models[trait] = model

print("\n Training complete. Models and vectorizer saved in current directory.")
